In [ ]:
today_date = "2026-08-09"

In [11]:
abfs_path = 'abfss://Fabric_Dev@onelake.dfs.fabric.microsoft.com/Fabric_LH_Sales.Lakehouse/Files/landing'


partition_path = f"/Processing_date={today_date}"

complete_path = abfs_path + partition_path

print(complete_path)

StatementMeta(, fc97fb9b-7ff4-4fd3-97f2-b0283828e0b6, 13, Finished, Available, Finished, False)

abfss://Fabric_Dev@onelake.dfs.fabric.microsoft.com/Fabric_LH_Sales.Lakehouse/Files/landing/Processing_date=2026-08-09


In [13]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType, ArrayType, DateType
)

StatementMeta(, fc97fb9b-7ff4-4fd3-97f2-b0283828e0b6, 15, Finished, Available, Finished, False)

In [39]:
v_schema = StructType([
    StructField("Row_ID", StringType(), True),
    StructField("Order_ID", StringType(), True),
    StructField("Order_Date", DateType(), True),
    StructField("Ship_Date", DateType(), True),
    StructField("Ship_Mode", StringType(), True),
    StructField("Customer_ID", StringType(), True),
    StructField("Customer_Name", StringType(), True),
    StructField("Segment", StringType(), True),
    StructField("Postal_Code", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Market", StringType(), True),
    StructField("Product_ID", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Sub_Category", StringType(), True),
    StructField("Product_Name", StringType(), True),
    StructField("Sales", DoubleType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Discount", DoubleType(), True),
    StructField("Profit", DoubleType(), True),
    StructField("Shipping_Cost", DoubleType(), True),
    StructField("Order_Priority", StringType(), True),
    StructField("Month", StringType(), True),
    StructField("Year", StringType(), True)
])

StatementMeta(, fc97fb9b-7ff4-4fd3-97f2-b0283828e0b6, 41, Finished, Available, Finished, False)

In [40]:
source_path = abfs_path

df = spark.read \
    .option("header", True) \
    .schema(v_schema) \
    .csv(source_path)

display(df)

StatementMeta(, fc97fb9b-7ff4-4fd3-97f2-b0283828e0b6, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f4557234-16f4-4e92-8560-0fa15bbf860a)

In [41]:
df.createOrReplaceTempView("t_new_data")

StatementMeta(, fc97fb9b-7ff4-4fd3-97f2-b0283828e0b6, 43, Finished, Available, Finished, False)

In [42]:
%%sql
    SELECT *
    FROM t_new_data


StatementMeta(, fc97fb9b-7ff4-4fd3-97f2-b0283828e0b6, 44, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 27 fields>

In [43]:
Fabric_tablesales_bronze = 'abfss://Fabric_Dev@onelake.dfs.fabric.microsoft.com/Fabric_LH_Sales.Lakehouse/Tables/dbo/tblsales_bronze'

try:
    spark.read.format('delta').load(Fabric_tablesales_bronze).createOrReplaceTempView('t_tablesales_bronze')

except:
    v_create_table = """
    CREATE TABLE IF NOT EXISTS tblsales_bronze (
        Row_ID STRING,
        Order_ID STRING,
        Order_Date DATE,
        Ship_Date DATE,
        Ship_Mode STRING,
        Customer_ID STRING,
        Customer_Name STRING,
        Segment STRING,
        Postal_Code STRING,
        City STRING,
        State STRING,
        Country STRING,
        Region STRING,
        Market STRING,
        Product_ID STRING,
        Category STRING,
        Sub_Category STRING,
        Product_Name STRING,
        Sales DOUBLE,
        Quantity INT,
        Discount DOUBLE,
        Profit DOUBLE,
        Shipping_Cost DOUBLE,
        Order_Priority STRING,
        Month STRING,
        Year STRING,
        processing_date DATE
    )
    """

    spark.sql(v_create_table)

    spark.read.format('delta') \
        .load(Fabric_tablesales_bronze) \
        .createOrReplaceTempView('t_tablesales_bronze')

StatementMeta(, fc97fb9b-7ff4-4fd3-97f2-b0283828e0b6, 45, Finished, Available, Finished, False)

In [44]:
%%sql
select * from t_tablesales_bronze

StatementMeta(, fc97fb9b-7ff4-4fd3-97f2-b0283828e0b6, 46, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 27 fields>

In [47]:
spark.sql("""
SELECT
    Row_ID,
    COUNT(*) AS cnt
FROM t_new_data
GROUP BY Row_ID
HAVING COUNT(*) > 1
ORDER BY cnt DESC
""").show(20, False)

StatementMeta(, fc97fb9b-7ff4-4fd3-97f2-b0283828e0b6, 49, Finished, Available, Finished, False)

+------+---+
|Row_ID|cnt|
+------+---+
+------+---+



In [48]:
sql_statement = f"""
MERGE INTO tblsales_bronze AS target
USING t_new_data AS source

ON target.Row_ID = source.Row_ID

WHEN MATCHED THEN
    UPDATE SET
        target.Order_ID = source.Order_ID,
        target.Order_Date = source.Order_Date,
        target.Ship_Date = source.Ship_Date,
        target.Ship_Mode = source.Ship_Mode,
        target.Customer_ID = source.Customer_ID,
        target.Customer_Name = source.Customer_Name,
        target.Segment = source.Segment,
        target.Postal_Code = source.Postal_Code,
        target.City = source.City,
        target.State = source.State,
        target.Country = source.Country,
        target.Region = source.Region,
        target.Market = source.Market,
        target.Product_ID = source.Product_ID,
        target.Category = source.Category,
        target.Sub_Category = source.Sub_Category,
        target.Product_Name = source.Product_Name,
        target.Sales = source.Sales,
        target.Quantity = source.Quantity,
        target.Discount = source.Discount,
        target.Profit = source.Profit,
        target.Shipping_Cost = source.Shipping_Cost,
        target.Order_Priority = source.Order_Priority,
        target.Month = source.Month,
        target.Year = source.Year,
        target.processing_date = '{today_date}'

WHEN NOT MATCHED THEN
    INSERT (
        Row_ID,
        Order_ID,
        Order_Date,
        Ship_Date,
        Ship_Mode,
        Customer_ID,
        Customer_Name,
        Segment,
        Postal_Code,
        City,
        State,
        Country,
        Region,
        Market,
        Product_ID,
        Category,
        Sub_Category,
        Product_Name,
        Sales,
        Quantity,
        Discount,
        Profit,
        Shipping_Cost,
        Order_Priority,
        Month,
        Year,
        processing_date
    )
    VALUES (
        source.Row_ID,
        source.Order_ID,
        source.Order_Date,
        source.Ship_Date,
        source.Ship_Mode,
        source.Customer_ID,
        source.Customer_Name,
        source.Segment,
        source.Postal_Code,
        source.City,
        source.State,
        source.Country,
        source.Region,
        source.Market,
        source.Product_ID,
        source.Category,
        source.Sub_Category,
        source.Product_Name,
        source.Sales,
        source.Quantity,
        source.Discount,
        source.Profit,
        source.Shipping_Cost,
        source.Order_Priority,
        source.Month,
        source.Year,
        '{today_date}'
    )
"""

spark.sql(sql_statement)

StatementMeta(, fc97fb9b-7ff4-4fd3-97f2-b0283828e0b6, 50, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [49]:
result = spark.sql(sql_statement)
display(result)

StatementMeta(, fc97fb9b-7ff4-4fd3-97f2-b0283828e0b6, 51, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c7aae140-ee57-4531-bf32-4bfb06047df1)

In [51]:
%%sql
select * from tblsales_bronze

StatementMeta(, fc97fb9b-7ff4-4fd3-97f2-b0283828e0b6, 53, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 27 fields>